# 7. Deep Learning

Let's have a damped harmonic oscillator:
$$\frac{d^2x}{dt^2} + 2\zeta\omega_0\frac{dx}{dt} + \omega_0^2 x = 0,$$

on a time domain $t \in [0, 1]\text{ s}$.
We will train two models to learn the dynamics: a purely data-driven Artificial Neural Network (ANN), and a Physics-Informed Neural Network (PINN).
The models are provided with sparse observation data ($N = 35$) only in the first half of the time domain ($t \in [0, 0.5]\text{ s}$).


How well can the models interpolate within the training window? How well can they extrapolate into the unseen domain ($t \in (0.5, 1.0]\text{ s}$)?
First, run the code as-is to see, how the models behave. Then play around with the hyperparameters like number of epochs, number of layers, number of training samples and see, how it influences the results.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

Prepare data:

In [ ]:
# --- 1. System Parameters ---
omega0 = 20.0       # Natural angular frequency (rad/s)
zeta = 0.05         # Damping ratio (< 1 for underdamped)
omega_d = omega0 * np.sqrt(1.0 - zeta**2)

# Time domain: 0 to 1 second
t_min, t_max = 0.0, 1.0
t_full = np.linspace(t_min, t_max, 500)

# Exact analytical displacement x(t)
def exact_solution(t, zeta, omega0, omega_d):
    decay = np.exp(-zeta * omega0 * t)
    oscillation = np.cos(omega_d * t) + (zeta * omega0 / omega_d) * np.sin(omega_d * t)
    return decay * oscillation

x_full = exact_solution(t_full, zeta, omega0, omega_d)

# --- 2. Sample Training Points ---
t_train_max = 0.5 * t_max # let's assume we took training data only from the first half of the time domain
n_train_points = 35

# Random uniform sampling in [0, 0.5]
np.random.seed(42)
t_train = np.sort(np.random.uniform(t_min, t_train_max, n_train_points))
x_train = exact_solution(t_train, zeta, omega0, omega_d)

plt.figure(figsize=(10, 4.5))
plt.plot(t_full, x_full, 'k-', linewidth=1.8, label='Exact solution $x(t)$', alpha=0.6)
plt.scatter(t_train, x_train, color='navy', edgecolors='k', s=45, zorder=4, label=f'Training data ($N={n_train_points}$)')
plt.axvline(t_train_max, color='tab:red', linestyle='-.', linewidth=1.8, label='Training boundary ($t = 0.5$ s)')
plt.axvspan(t_min, t_train_max, color='tab:red', alpha=0.08, label='Training domain (Interpolation)')
plt.axvspan(t_train_max, t_max, color='tab:gray', alpha=0.08, label='Test domain (Extrapolation)')

plt.title('Damped Harmonic Oscillator: Ground Truth & Training Dataset')
plt.xlabel('t / s')
plt.ylabel('Displacement x / m')
plt.xlim([t_min, t_max])
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper right', framealpha=0.9)
plt.tight_layout()
plt.show()

print(f"Full dataset shape   : {t_full.shape[0]} points")

# Convert to PyTorch tensors for subsequent NN steps
t_train_tensor = torch.tensor(t_train, dtype=torch.float32).unsqueeze(1)
x_train_tensor = torch.tensor(x_train, dtype=torch.float32).unsqueeze(1)
t_full_tensor = torch.tensor(t_full, dtype=torch.float32).unsqueeze(1)
print(f"Training tensor shape: {t_train_tensor.shape}")


Shared Hyperparameters & Architecture

In [ ]:
hidden_dim = 32
Activation = nn.Tanh
fnn_epochs = 4000
adam_epochs = 2000      # PINN First phase: Adam to find the general basin
lbfgs_epochs = 2000     # PINN Second phase: L-BFGS for high-precision convergence
lr = 1e-3
lambda_physics = 1

def build_model():
    return nn.Sequential(
        nn.Linear(1, hidden_dim),
        Activation(),
        nn.Linear(hidden_dim, hidden_dim),
        Activation(),
        nn.Linear(hidden_dim, 1)
    )

# Collocation points across full time domain [0, 1.0] for PINN
n_collocation = 25
t_physics = np.linspace(t_min, t_max, n_collocation)
t_physics_tensor = torch.tensor(t_physics, dtype=torch.float32).unsqueeze(1)

Run the training:

In [ ]:
# ==========================================
# 1. Train ANN (Data-Only)
# ==========================================
torch.manual_seed(42)
fnn_model = build_model()
fnn_optimizer = torch.optim.Adam(fnn_model.parameters(), lr=lr)
loss_fn = nn.MSELoss()

fnn_loss_history = []
print("--- Training ANN ---")
for epoch in range(fnn_epochs):
    fnn_optimizer.zero_grad()
    pred_data = fnn_model(t_train_tensor)
    loss = loss_fn(pred_data, x_train_tensor)
    loss.backward()
    fnn_optimizer.step()
    
    fnn_loss_history.append(loss.item())

    if (epoch + 1) % 1000 == 0:
        print(f"ANN  | Epoch [{epoch+1:4d}/{fnn_epochs}] | Loss: {loss.item():.6e}")

# ==========================================
# 2. Train PINN (Adam + L-BFGS)
# ==========================================
torch.manual_seed(42)  # Identical weight initialization
pinn_model = build_model()

pinn_total_history = []
pinn_data_history = []
pinn_physics_history = []

def compute_ode_residual(model, t, zeta, omega0):
    t.requires_grad_(True)
    x = model(t)
    dx_dt = torch.autograd.grad(
        outputs=x, inputs=t,
        grad_outputs=torch.ones_like(x),
        create_graph=True
    )[0]
    d2x_dt2 = torch.autograd.grad(
        outputs=dx_dt, inputs=t,
        grad_outputs=torch.ones_like(dx_dt),
        create_graph=True
    )[0]
    raw_residual = d2x_dt2 + 2.0 * zeta * omega0 * dx_dt + (omega0**2) * x
    return raw_residual / (omega0**2)

# --- Phase A: Adam Optimizer ---
print(f"--- Training PINN: Phase 1 (Adam, {adam_epochs} epochs) ---")
pinn_adam = torch.optim.Adam(pinn_model.parameters(), lr=lr)

for epoch in range(adam_epochs):
    pinn_adam.zero_grad()
    
    # Data loss
    pred_data = pinn_model(t_train_tensor)
    loss_data = loss_fn(pred_data, x_train_tensor)
    
    # Physics residual loss
    residual = compute_ode_residual(pinn_model, t_physics_tensor, zeta, omega0)
    loss_physics = loss_fn(residual, torch.zeros_like(residual))
    
    total_loss = loss_data + lambda_physics * loss_physics
    total_loss.backward()
    pinn_adam.step()
    
    pinn_total_history.append(total_loss.item())
    pinn_data_history.append(loss_data.item())
    pinn_physics_history.append(loss_physics.item())

    if (epoch + 1) % 500 == 0:
        print(f"Adam  | Epoch [{epoch+1:4d}/{adam_epochs}] | Total: {total_loss.item():.6e} | Data: {loss_data.item():.6e} | Physics: {loss_physics.item():.6e}")

# --- Phase B: L-BFGS Optimizer ---
print(f"--- Training PINN: Phase 2 (L-BFGS, max {lbfgs_epochs} iterations) ---")
pinn_lbfgs = torch.optim.LBFGS(
    pinn_model.parameters(),
    lr=1.0,
    max_iter=lbfgs_epochs,
    max_eval=int(lbfgs_epochs * 1.25),
    history_size=50,
    tolerance_grad=1e-7,
    tolerance_change=1e-9,
    line_search_fn="strong_wolfe"
)

iter_count = [0]

def closure():
    pinn_lbfgs.zero_grad()
    
    pred_data = pinn_model(t_train_tensor)
    loss_data = loss_fn(pred_data, x_train_tensor)
    
    residual = compute_ode_residual(pinn_model, t_physics_tensor, zeta, omega0)
    loss_physics = loss_fn(residual, torch.zeros_like(residual))
    
    total_loss = loss_data + lambda_physics * loss_physics
    total_loss.backward()
    
    # Log per-iteration loss history seamlessly
    pinn_total_history.append(total_loss.item())
    pinn_data_history.append(loss_data.item())
    pinn_physics_history.append(loss_physics.item())
    
    iter_count[0] += 1
    if iter_count[0] % 200 == 0:
        print(f"LBFGS | Iter  [{iter_count[0]:4d}] | Total: {total_loss.item():.6e} | Data: {loss_data.item():.6e} | Physics: {loss_physics.item():.6e}")
        
    return total_loss

pinn_lbfgs.step(closure)
print(f"L-BFGS finished after {iter_count[0]} function evaluations.")

In [ ]:
fnn_steps = np.arange(1, len(fnn_loss_history) + 1)
pinn_steps = np.arange(1, len(pinn_total_history) + 1)

plt.figure(figsize=(12, 4.5))
plt.subplot(1, 2, 1)
plt.semilogy(fnn_steps, fnn_loss_history, color='crimson', linewidth=1.8, label=r'ANN Total Loss ($\mathcal{L}_{\mathrm{data}}$)')
plt.semilogy(pinn_steps, pinn_total_history, color='royalblue', linewidth=1.8, label=r'PINN Total Loss ($\mathcal{L}_{\mathrm{total}}$)')
plt.axvline(adam_epochs, color='gray', linestyle=':', linewidth=1.2, label='L-BFGS Switch')
plt.title('Training Loss: FNN vs. PINN')
plt.xlabel('Optimization Step / Iteration')
plt.ylabel('Loss (MSE)')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper right', framealpha=0.9)

plt.subplot(1, 2, 2)
plt.semilogy(pinn_steps, pinn_data_history, color='navy', linestyle='-', linewidth=1.8, label=r'Data Loss ($\mathcal{L}_{\mathrm{data}}$)')
plt.semilogy(pinn_steps, pinn_physics_history, color='tab:orange', linestyle='--', linewidth=1.8, label=r'Physics Residual ($\mathcal{L}_{\mathrm{physics}}$)')
plt.semilogy(pinn_steps, pinn_total_history, color='royalblue', linestyle=':', linewidth=2.0, label=r'Combined ($\mathcal{L}_{\mathrm{total}}$)')
plt.axvline(adam_epochs, color='gray', linestyle=':', linewidth=1.2, label='L-BFGS Switch')
plt.title('PINN Loss Component Breakdown')
plt.xlabel('Optimization Step / Iteration')
plt.ylabel('Loss (MSE)')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper right', framealpha=0.9)

plt.tight_layout()
plt.show()

In [ ]:
# --- 1. Evaluate on Full Domain ---
fnn_model.eval()
pinn_model.eval()

with torch.no_grad():
    x_pred_fnn = fnn_model(t_full_tensor).numpy().flatten()
    x_pred_pinn = pinn_model(t_full_tensor).numpy().flatten()

# --- 2. Split into Interpolation and Extrapolation Regimes ---
interp_mask = t_full <= t_train_max
extrap_mask = t_full > t_train_max

# FNN Errors
fnn_mse_interp = np.mean((x_pred_fnn[interp_mask] - x_full[interp_mask])**2)
fnn_max_interp = np.max(np.abs(x_pred_fnn[interp_mask] - x_full[interp_mask]))
fnn_mse_extrap = np.mean((x_pred_fnn[extrap_mask] - x_full[extrap_mask])**2)
fnn_max_extrap = np.max(np.abs(x_pred_fnn[extrap_mask] - x_full[extrap_mask]))

# PINN Errors
pinn_mse_interp = np.mean((x_pred_pinn[interp_mask] - x_full[interp_mask])**2)
pinn_max_interp = np.max(np.abs(x_pred_pinn[interp_mask] - x_full[interp_mask]))
pinn_mse_extrap = np.mean((x_pred_pinn[extrap_mask] - x_full[extrap_mask])**2)
pinn_max_extrap = np.max(np.abs(x_pred_pinn[extrap_mask] - x_full[extrap_mask]))

# --- 3. Print Metrics Table ---
print("=" * 68)
print(f"{'Metric / Domain':<28} | {'Standard ANN':<16} | {'PINN':<16}")
print("=" * 68)
print(f"{'Interpolation MSE [0, 0.5]':<28} | {fnn_mse_interp:<16.4e} | {pinn_mse_interp:<16.4e}")
print(f"{'Interpolation Max Error':<28} | {fnn_max_interp:<16.4e} | {pinn_max_interp:<16.4e}")
print("-" * 68)
print(f"{'Extrapolation MSE (0.5, 1.0]':<28} | {fnn_mse_extrap:<16.4e} | {pinn_mse_extrap:<16.4e}")
print(f"{'Extrapolation Max Error':<28} | {fnn_max_extrap:<16.4e} | {pinn_max_extrap:<16.4e}")
print("=" * 68)

plt.figure(figsize=(15, 4.5))

# Left: ANN
plt.subplot(1, 2, 1)
plt.plot(t_full, x_full, 'k-', linewidth=1.8, label='Exact solution $x(t)$', alpha=0.6)
plt.plot(t_full, x_pred_fnn, color='crimson', linestyle='--', linewidth=2.2, label='ANN prediction')
plt.scatter(t_train, x_train, color='navy', edgecolors='k', s=45, zorder=4, label=f'Training data ($N={n_train_points}$)')
plt.axvline(t_train_max, color='tab:red', linestyle='-.', linewidth=1.8, label='Training boundary ($t = 0.5$ s)')
plt.axvspan(t_min, t_train_max, color='tab:red', alpha=0.08)
plt.axvspan(t_train_max, t_max, color='tab:gray', alpha=0.08)
plt.title(f'ANN (Extrap MSE: {fnn_mse_extrap:.2e})')
plt.xlabel('t / s')
plt.ylabel('Displacement x / m')
plt.xlim([t_min, t_max])
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper right', framealpha=0.9)

# Right: PINN
plt.subplot(1, 2, 2)
plt.plot(t_full, x_full, 'k-', linewidth=1.8, label='Exact solution $x(t)$', alpha=0.6)
plt.plot(t_full, x_pred_pinn, color='royalblue', linestyle='--', linewidth=2.2, label='PINN prediction')
plt.scatter(t_train, x_train, color='navy', edgecolors='k', s=45, zorder=4, label=f'Training data ($N={n_train_points}$)')
plt.axvline(t_train_max, color='tab:red', linestyle='-.', linewidth=1.8, label='Training boundary ($t = 0.5$ s)')
plt.axvspan(t_min, t_train_max, color='tab:red', alpha=0.08)
plt.axvspan(t_train_max, t_max, color='tab:gray', alpha=0.08)
plt.title(f'PINN (Extrap MSE: {pinn_mse_extrap:.2e})')
plt.xlabel('t / s')
plt.xlim([t_min, t_max])
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper right', framealpha=0.9)

plt.tight_layout()
plt.show()